<div align="center">

# 🚕 Yandex Go Analytics

## 7 · Refactor via Delta Live Tables

*The same medallion logic, rewritten declaratively with data-quality expectations.*

![engine](https://img.shields.io/badge/engine-Delta%20Live%20Tables-1B3139)
![quality](https://img.shields.io/badge/quality-Expectations-2E7D32)
![compute](https://img.shields.io/badge/compute-Serverless-FF3621)
![deploy](https://img.shields.io/badge/deploy-Asset%20Bundles-4B8BBE)

</div>

| ◀ Previous | 🏠 Home | Next ▶ |
|:--|:--:|--:|
| [6 · Data Enrichment](./6.%20Data%20Enrichment.ipynb) | [Project Overview](./0.%20Project%20Overview.ipynb) | [8 · Security, Governance](./8.%20Security%2C%20Governance.ipynb) |

**Progress** &nbsp; ▰▰▰▰▰▰▰▱  **7 / 8**

---

## 🎯 Task requirements

Recreate the pipeline with DLT syntax: *Jobs & Pipelines → Create Pipeline* and a DLT notebook that replicates the logic.

- [x] **Bronze** — streaming tables (`@dlt.table`) reading the raw trips, drivers and users
- [x] **Silver** — data-quality expectations (`@dlt.expect…_or_drop`) filter bad records; enrich with text statuses
- [x] **Gold** — aggregates (trips and revenue per zone) and the trips ⨝ driver-profiles join

## 📂 The pipeline source code

The pipeline is made of three notebooks that live next to the project code:

| Layer | Notebook | Contains |
|:--:|:--|:--|
| 🥉 Bronze | [`bronze.ipynb`](../dlt_pipelines/yandex_go_trips_pipeline/bronze.ipynb) | Streaming tables over Auto Loader: `bronze_taxi`, `bronze_users`, `bronze_drivers` |
| 🥈 Silver | [`silver.ipynb`](../dlt_pipelines/yandex_go_trips_pipeline/silver.ipynb) | Expectations + enrichment: `silver_taxi`, `silver_drivers`, `silver_users` |
| 🥇 Gold | [`gold.ipynb`](../dlt_pipelines/yandex_go_trips_pipeline/gold.ipynb) | `gold_taxi_metrics`, `gold_driver_metrics`, `gold_user_metrics`, `gold_enriched_trips` |

⚙️ Pipeline definition: [`resources/pipelines/yandex_go_trips_pipeline.yml`](../../resources/pipelines/yandex_go_trips_pipeline.yml)
&nbsp;·&nbsp; ⏰ Scheduled job: [`resources/jobs/scheduled_tripes_pipeline_job.yml`](../../resources/jobs/scheduled_tripes_pipeline_job.yml)
&nbsp;·&nbsp; 🛠 Commands: [`Makefile`](../../Makefile)

## 🧭 From imperative services to a declarative pipeline

| Concern | Notebooks 2 – 6 (Python services) | This notebook (DLT) |
|:--|:--|:--|
| Orchestration | You call `service.run(spark)` in order | DLT builds the dependency graph from `dlt.read` / `dlt.read_stream` |
| Checkpoints | Managed by hand (`checkpointLocation`) | Managed by the pipeline |
| Data quality | `DataFrame.filter(...)` | **Expectations**, with pass / drop counts recorded automatically |
| Retries & monitoring | Your code | Built-in event log and UI lineage |
| Target | `bronze` / `silver` / `gold` schemas | One schema, `trips_dlt` (`bronze_*`, `silver_*`, `gold_*` tables) |

### 🕸 Pipeline graph

```text
 landing/parquet ─▶ bronze_taxi ────▶ silver_taxi ─────┬─▶ gold_taxi_metrics
                                                       └─▶ gold_enriched_trips ◀─┐
 landing/json ────▶ bronze_drivers ─▶ silver_drivers ──┬─▶ gold_driver_metrics   │ LEFT JOIN
                                                       └─────────────────────────┘
 landing/avro ────▶ bronze_users ───▶ silver_users ────▶ gold_user_metrics
```

## 🥉 Bronze — streaming tables

One small factory builds the three tables; Auto Loader (`cloudFiles`) streams the files and adds the audit columns.

```python
@dlt.table(name=self.table_name, comment=f"Raw streaming ingestion for {self.table_name}")
def _table_func():
    reader = spark.readStream.format("cloudFiles").option("cloudFiles.format", self.source_format)
    if self.enable_schema_evolution:                       # drivers only
        reader = reader.option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    return (
        reader.load(self.source_path)
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
    )
```

## 🥈 Silver — expectations

Rows that break an expectation are **dropped**, and the pipeline records how many.

| Table | Expectation | Rule | On violation |
|:--|:--|:--|:--:|
| `silver_taxi` | `valid_passenger_count` | `passenger_count > 0` | 🗑 drop |
| `silver_taxi` | `valid_trip_distance` | `trip_distance > 0` | 🗑 drop |
| `silver_taxi` | `valid_fare_amount` | `fare_amount > 0` | 🗑 drop |
| `silver_drivers` | `valid_experience` | `experience > 0` | 🗑 drop |
| `silver_drivers` | `valid_rating_range` | `rating >= 1.0 AND rating <= 5.0` | 🗑 drop |
| `silver_users` | `valid_full_name` | `full_name IS NOT NULL` | 🗑 drop |

Enrichment: `payment_type_description` (*Credit Card* / *Cash* / *Other*), `_processed_at`, de-duplication.

```python
@dlt.table(name="silver_taxi")
@dlt.expect_all_or_drop(TripSilverPipeline.DATA_QUALITY_RULES)   # dict of rules, all "or drop"
def silver_taxi():
    return TripSilverPipeline.clean(dlt.read_stream("bronze_taxi"))
```

> 📝 `@dlt.expect_all_or_drop` is the dictionary form of `@dlt.expect_or_drop`: same behaviour, several rules at once.

## 🥇 Gold — metrics and enrichment

| Table | Logic |
|:--|:--|
| `gold_taxi_metrics` | trips and revenue per pickup zone and payment type |
| `gold_driver_metrics` | drivers count and average rating per experience level |
| `gold_user_metrics` | registrations per day |
| `gold_enriched_trips` | `silver_taxi` **LEFT JOIN** `silver_drivers` on `driver_id` |

## 🚀 Create and run the pipeline

### Option A · Databricks UI &nbsp;<sub>(*Jobs & Pipelines → Create → ETL pipeline*)</sub>

1. Open **Jobs & Pipelines** and click **Create → ETL pipeline**.
2. Fill in the basics:

   | Setting | Value |
   |:--|:--|
   | Name | `Yandex Go DLT Pipeline for Trips` |
   | Catalog | your catalog, e.g. `yandex_go_dev` |
   | Target schema | `trips_dlt` |
   | Compute | ☑ Serverless |
   | Source code | `bronze.ipynb`, `silver.ipynb`, `gold.ipynb` from `src/dlt_pipelines/yandex_go_trips_pipeline/` |
   | Configuration | `pipeline.catalog` = your catalog |

3. Click **Start** — DLT resolves the graph and runs bronze → silver → gold.

### Option B · Databricks Asset Bundles <sub>(recommended — reproducible)</sub>

```bash
make deploy-dev        # databricks bundle deploy -t dev      (creates / updates the pipeline)
make run-trips-dev     # databricks bundle run yandex_go_trips_pipeline -t dev

# or in one go
make deploy-and-run-trips
```

> 💡 Pipeline settings (serverless, `advanced` edition, triggered mode, `pipeline.catalog` configuration) are
> defined once in `resources/pipelines/yandex_go_trips_pipeline.yml`. The daily job
> `[dev] Scheduled Trips Pipeline Run` triggers the same pipeline at 00:00 UTC.

---

## ✅ Checkpoint

- [x] Bronze streaming tables declared with `@dlt.table`
- [x] Silver data quality enforced with `@dlt.expect_all_or_drop` expectations
- [x] Gold aggregates and the trips ⨝ drivers join declared in DLT
- [x] Pipeline reproducibly deployed and run with Databricks Asset Bundles

> **Next up ▶ [8 · Security, Governance](./8.%20Security%2C%20Governance.ipynb)** — secure the Gold schema, audit the workflow and package everything as code.

<div align="center"><sub>Yandex Go Analytics · notebook 7 of 8 · <a href="https://github.com/stanislavpanfilenko/YandexGoAnalytics">GitHub</a>